In [1]:
# Make this notebook work from either fine-tuning/ or fine-tuning/live-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())

cwd: C:\Users\okofoworola\Acme Health Demo\fine-tuning


# Lab 09 · Foundry Decision Advisor — the capstone (LIVE)

> **LIVE DEMO LAB.** Tracing is sent under service name `acme-live-demo` so this run shows up
> alongside the other live labs in the Foundry **Tracing** tab. With Azure creds present the
> advisor adds a live LLM classification pass on top of the offline heuristics.

An experienced developer has to decide whether Foundry is worth it — but given a workload, it's hard to tell which model fits or which capabilities it actually needs. Drop in code or a task and the advisor reasons it out: it recommends a model from the live catalog with a rationale and a cost / latency / accuracy tradeoff, maps each gap to a Foundry capability (SFT, DPO, tool-FT, RAG, memory, dates, eval, guardrails, routing, tracing) and the lab that proves it, and emits a structured decision trace — `trace_id`, model, tokens, flags, confidence — to Application Insights. Paste a first-draft RAG chatbot and watch six gaps get flagged and routed to the right lab, with an auditable trace. *Foundry isn't an endpoint — it's the platform that tells you what to build.*

---
## Step 0 — Enable Foundry tracing (live)

In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='acme-live-demo')

[tracing] enabled. Service name : acme-live-demo
[tracing] Open Foundry portal -> your project -> Tracing tab
[tracing] (also visible under App Insights: appi-shuttervoice-dev)


True

---
## Step 1 — Load the advisor + Foundry model catalog

In [3]:
from _advisor import (
    FoundryDecisionAdvisor,
    load_catalog,
    try_build_client,
    print_report,
)
import pandas as pd

catalog = load_catalog()
print(f'Loaded {len(catalog)} Foundry deployments from the catalog:')
pd.DataFrame(catalog)[['deployment', 'tier', 'fine_tunable', 'relative_cost', 'relative_latency', 'reasoning_score']]

Loaded 7 Foundry deployments from the catalog:


,deployment,tier,fine_tunable,relative_cost,relative_latency,reasoning_score
0,gpt-4o,reasoning,False,8,6,9
1,gpt-4o-mini,fast,True,2,2,6
2,model-router,router,False,4,4,8
3,text-embedding-3-large,embedding,False,1,1,0
4,acme-sft-deployment,fine_tuned,False,2,2,6
5,acme-dpo-deployment,fine_tuned,False,2,2,6
6,acme-tools-deployment,fine_tuned,False,2,2,6


In [4]:
# Live mode: build the Azure client so the classifier gets an LLM refinement pass.
client = try_build_client()
advisor = FoundryDecisionAdvisor(client=client)
print('Mode:', 'LIVE (Azure)' if client else 'MOCK (offline)')

Mode: LIVE (Azure)


---
## Step 2 — “Connect your code”: analyze a real code sample

*A developer's first-draft RAG chatbot — one hardcoded model, no eval, no guardrails, weekly-changing
knowledge. Watch the advisor map every gap to a Foundry capability + lab.*

In [5]:
sample_code = Path('data/samples/rag_chatbot.py').read_text(encoding='utf-8')
print(sample_code)

# SAMPLE INPUT for Lab 09 — a typical "first draft" RAG chatbot.
# Drop a file like this into the advisor and watch it recommend Foundry features.
from openai import AzureOpenAI

client = AzureOpenAI()

def answer_member_question(question: str) -> str:
    # One hardcoded model for everything — easy + hard requests alike.
    model = "gpt-4o"

    # Pulls from the Acme formulary / knowledge base, which changes weekly.
    docs = retrieve_from_knowledge_base(question)

    prompt = f"Answer using these docs:\n{docs}\n\nQuestion: {question}"
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content



In [6]:
result = advisor.analyze(sample_code)
print_report(result)

  FOUNDRY DECISION ADVISOR  ·  trace 00c0015e

  Detected workload : rag
  All task types    : rag, extraction, summarization
  Classified via    : heuristic+llm

  ── Recommended model ──────────────────────────────────────────
  → text-embedding-3-large  (tier: embedding, score 10/10)
    Why     : 'rag' needs embeddings for grounding. text-embedding-3-large produces the vectors; pair it with gpt-4o-mini to answer from retrieved chunks.
    Tradeoff: Lowest cost/latency; not a chat model — must be paired with a generator.

  ── Foundry capabilities to adopt ──────────────────────────────
  • Model Routing  →  09_foundry_decision_advisor.ipynb
      need: hardcoded_model  |  Route each request to the cheapest model that can handle it instead of one hardcoded model.
  • Evaluation  →  07_evaluation.ipynb
      need: no_evaluation  |  Replace 'felt better today' with keyword + LLM-judge scoreboards per release.
  • Guardrails  →  08_guardrails.ipynb
      need: no_guardrails  |  Add the

---
## Step 3 — Model routing with rationale

*Four different workloads — the router picks a different model for each and explains the
cost/latency/accuracy tradeoff.*

In [7]:
workloads = {
    'Nightly summarizer (10k transcripts, cost-sensitive)':
        'Summarize 10,000 member call transcripts every night into 3-bullet digests. Cost matters.',
    'Clinical reasoning over member history':
        'Reason step-by-step about medical necessity and diagnosis/procedure alignment from clinical notes.',
    'High-volume tool-calling agent':
        Path('data/samples/tool_calling_agent.py').read_text(encoding='utf-8'),
    'Grounded answer from changing formulary (RAG)':
        'Answer member questions grounded in the Acme formulary knowledge base, which changes weekly.',
}

rows = []
for title, text in workloads.items():
    cost = 'cost' in title.lower() or 'summar' in title.lower()
    r = advisor.analyze(text, prefer_cost=cost)
    rows.append({
        'workload': title,
        'task': r.primary_task,
        'model': r.model.deployment,
        'score': r.model.score,
        'method': r.classification_method,
    })

pd.DataFrame(rows)

,workload,task,model,score,method
0,"Nightly summarizer (10k transcripts, cost-sens...",summarization,gpt-4o-mini,8,heuristic+llm
1,Clinical reasoning over member history,clinical_reasoning,gpt-4o,9,heuristic+llm
2,High-volume tool-calling agent,tool_calling,acme-tools-deployment,9,heuristic+llm
3,Grounded answer from changing formulary (RAG),extraction,gpt-4o-mini,8,heuristic+llm


---
## Step 4 — The decision trace flows to Foundry Tracing

*Structured trace prints as JSON and (with tracing enabled above) lands in App Insights → the
Foundry **Tracing** tab. Open the portal during the demo to show the span live.*

In [8]:
import json
from dataclasses import asdict

r = advisor.analyze(sample_code)
print(json.dumps(asdict(r.trace), indent=2))

{
  "trace_id": "bd6f9a06-2f4c-410a-bfee-2b00b005cc14",
  "task_types": [
    "rag",
    "extraction",
    "summarization"
  ],
  "primary_task": "rag",
  "classification_method": "heuristic+llm",
  "model_selected": "text-embedding-3-large",
  "model_score": 10,
  "feature_count": 6,
  "prompt_tokens_estimate": 179,
  "latency_ms": 1124.25,
  "flags": [
    "hardcoded_model",
    "no_evaluation",
    "no_guardrails",
    "needs_domain_facts",
    "external_knowledge",
    "no_observability"
  ],
  "confidence": 0.89
}


---
## Step 5 — Mini evaluation: does the advisor route correctly?

In [9]:
EVAL_CASES = [
    ('Summarize call transcripts into bullets', 'summarization', {'gpt-4o-mini', 'acme-sft-deployment'}),
    ('tools=[...]; client.chat.completions.create(tools=tools)', 'tool_calling', {'acme-tools-deployment', 'gpt-4o'}),
    ('Retrieve from knowledge base with embeddings and cosine similarity', 'rag', {'text-embedding-3-large'}),
    ('Detect PII and block jailbreak attempts for safety', 'safety_review', {'gpt-4o', 'model-router'}),
]

# Regression check runs on the deterministic heuristic engine (client=None) so the
# assertion is reproducible. The optional LLM refinement is showcased in Step 3.
eval_advisor = FoundryDecisionAdvisor(client=None)

passed = 0
for text, expected_task, ok_models in EVAL_CASES:
    r = eval_advisor.analyze(text)
    task_ok = r.primary_task == expected_task
    model_ok = r.model.deployment in ok_models
    status = 'PASS' if (task_ok and model_ok) else 'FAIL'
    passed += task_ok and model_ok
    print(f'[{status}] task={r.primary_task:<18} model={r.model.deployment:<24} ({text[:40]}...)')

print(f'\n{passed}/{len(EVAL_CASES)} eval cases passed.')
assert passed == len(EVAL_CASES), 'Advisor routing regressed!'


[PASS] task=summarization      model=gpt-4o-mini              (Summarize call transcripts into bullets...)
[PASS] task=tool_calling       model=acme-tools-deployment  (tools=[...]; client.chat.completions.cre...)
[PASS] task=rag                model=text-embedding-3-large   (Retrieve from knowledge base with embedd...)
[PASS] task=safety_review      model=gpt-4o                   (Detect PII and block jailbreak attempts ...)

4/4 eval cases passed.


---
## Takeaways

- **Model choice is a justified decision**, not a hardcoded constant.
- **Every gap maps to a Foundry capability** proven by a lab in this repo (00–08).
- **Governed + observable**: traces land in App Insights → Foundry Tracing.
- **No lock-in to evaluate**: runs offline in mock mode, swaps cleanly to live Foundry calls.